# Pipeline Results
End-to-end summary of the infrastructure imagery curation pipeline.
Shows funnel metrics, tile quality, and example outputs.

Run the full pipeline first:
```bash
python pipeline.py
```
Or run with a small sample:
```bash
# in pipeline.py, set SAMPLE_PER_TYPE = 3, then:
python pipeline.py
```

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from imagery import stretch_rgb
from dataset import DatasetAssembler

## Configuration

In [ ]:
DATASET_DIR  = '../data/dataset_maine_v1'
ASSETS_CSV   = '../data/maine_all_assets.csv'
DEDUPED_CSV  = '../data/maine_deduped_assets.csv'

## Pipeline funnel

In [ ]:
# Load counts from each stage
df_raw    = pd.read_csv(ASSETS_CSV)   if os.path.exists(ASSETS_CSV)   else pd.DataFrame()
df_deduped = pd.read_csv(DEDUPED_CSV) if os.path.exists(DEDUPED_CSV) else pd.DataFrame()

assembler = DatasetAssembler(DATASET_DIR)
manifest  = assembler.load_manifest()
df_dataset = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('bbox','image_shape')}
    for r in manifest['records']
])

stages = {
    'Extracted\n(OSM)':       len(df_raw),
    'After\nDedup':            len(df_deduped),
    'Tiles\nFetched':          manifest['n_tiles'],
    'QC\nPassed':              manifest['n_tiles'],  # update if you track separately
    'Triage\nAccepted':        len(df_dataset),
}

fig, ax = plt.subplots(figsize=(10, 4))
colors  = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']
bars    = ax.bar(stages.keys(), stages.values(), color=colors, width=0.5)

for bar, val in zip(bars, stages.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stages.values())*0.01,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Pipeline Funnel — Assets Through Each Stage', fontsize=13, pad=12)
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('pipeline_funnel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pipeline_funnel.png')

## Asset type breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw extraction
if not df_raw.empty:
    counts = df_raw['asset_type'].value_counts()
    axes[0].barh(counts.index, counts.values, color='#4C72B0')
    axes[0].set_title('Extracted assets by type')
    axes[0].set_xlabel('Count')
    axes[0].spines[['top','right']].set_visible(False)

# Dataset tiles
if not df_dataset.empty:
    counts2 = df_dataset['asset_type'].value_counts()
    axes[1].barh(counts2.index, counts2.values, color='#55A868')
    axes[1].set_title('Dataset tiles by type')
    axes[1].set_xlabel('Count')
    axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Asset Type Distribution', fontsize=13)
plt.tight_layout()
plt.savefig('asset_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## Source breakdown

In [ ]:
if not df_dataset.empty:
    source_counts = df_dataset['source'].value_counts()
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(source_counts.index, source_counts.values,
           color=['#4C72B0','#55A868','#C44E52'][:len(source_counts)])
    ax.set_title('Dataset tiles by imagery source')
    ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)
    for i, (idx, val) in enumerate(source_counts.items()):
        ax.text(i, val + max(source_counts)*0.01, str(val),
                ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('source_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()

## Confidence distribution

In [ ]:
if not df_dataset.empty and 'confidence' in df_dataset.columns:
    conf_counts = df_dataset['confidence'].value_counts()
    colors_conf = {'high': '#55A868', 'low': '#CCB974', 'contradiction': '#C44E52'}
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(conf_counts.index,
           conf_counts.values,
           color=[colors_conf.get(c, '#4C72B0') for c in conf_counts.index])
    ax.set_title('Dataset tiles by confidence level')
    ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()

## Example tiles — one grid per asset type

In [ ]:
asset_types = df_dataset['asset_type'].unique()

for asset_type in sorted(asset_types):
    type_records = [r for r in manifest['records']
                    if r['asset_type'] == asset_type][:6]
    if not type_records:
        continue

    n      = len(type_records)
    ncols  = min(3, n)
    nrows  = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4*ncols, 4*nrows))
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for ax, record in zip(axes, type_records):
        try:
            image = assembler.load_tile(record)
            ax.imshow(stretch_rgb(image))
        except Exception:
            ax.text(0.5, 0.5, 'load error', ha='center', va='center',
                    transform=ax.transAxes)
        ax.set_title(
            f"{record['source']}\n{record['image_date']}",
            fontsize=7,
        )
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle(asset_type, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(f"tiles_{asset_type.replace('.','_')}.png",
                dpi=150, bbox_inches='tight')
    plt.show()

## Dataset summary table

In [ ]:
print(f'Dataset: {DATASET_DIR}')
print(f'Created: {manifest["created_at"]}')
print(f'Total tiles: {manifest["n_tiles"]}')
print()
df_dataset[['asset_type','source','image_date','confidence']].head(20)

## Pipeline yield rate

In [ ]:
if len(df_raw) > 0 and len(df_dataset) > 0:
    print('Pipeline yield rates:')
    print(f'  Extraction → dataset:  {len(df_dataset)/len(df_raw)*100:.1f}%')
    if len(df_deduped) > 0:
        print(f'  After dedup → dataset: {len(df_dataset)/len(df_deduped)*100:.1f}% of imagery attempts')
    print()
    print('Tiles per asset type in final dataset:')
    print(df_dataset.groupby(["asset_type","source"]).size().unstack(fill_value=0).to_string())